# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**All entities are referenced by their `@id`.**

In [ ]:
# Print available record sets and fields by @id

record_sets = dataset.record_sets()
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")
    fields = dataset.fields(record_set=rs['@id'])
    print("  Fields:")
    for field in fields:
        print(f"    - Field @id: {field['@id']} | Name: {field.get('name', '(no name)')} | Data type: {field.get('dataType', '(unknown)')}")
    print()
    # Show example records
    records = list(dataset.records(record_set=rs['@id']))
    if records:
        print(f"  Example records from {rs['@id']}:\n    {records[0]}")
    else:
        print("  No records found.")
    print("="*60)


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all data from each available record set

# Get the list of record_set @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame for RecordSet @id {record_set_id}: columns -> {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records available for RecordSet @id {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Use the relevant `@id` for numeric fields and grouping.

In [ ]:
# Example EDA: Filter numeric values, normalize, and group

# Choose a record set to analyze (use the first one if available)
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes.get(rs_id)
    if df is not None and not df.empty:
        # Identify numeric fields by @id and name (from metadata)
        fields = dataset.fields(record_set=rs_id)
        numeric_field_id = None
        # Find the first numeric field (e.g., Integer or Float)
        for field in fields:
            if field.get('dataType') in ['Integer', 'Float', 'Number']:
                numeric_field_id = field['@id']
                numeric_field_name = field.get('name', numeric_field_id)
                break
        if numeric_field_id is not None and numeric_field_id in df.columns:
            threshold = df[numeric_field_id].quantile(0.75)   # Use 75th percentile as threshold
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold} (field name: {numeric_field_name}):")
            print(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by a non-numeric field (like anatomical location, if available)
            # Find a likely categorical field
            group_field_id = None
            for field in fields:
                if field.get('dataType') in ['Text']:
                    group_field_id = field['@id']
                    group_field_name = field.get('name', group_field_id)
                    break
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (field name: {group_field_name}): Mean of {numeric_field_id}")
                print(grouped_df.head())
            else:
                print("No suitable categorical field found for grouping.")
        else:
            print("No numeric field found in the current DataFrame.")
    else:
        print("No DataFrame available for EDA.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Use pandas and matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot if numeric and group fields are available
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes.get(rs_id)
    if df is not None and not df.empty:
        fields = dataset.fields(record_set=rs_id)
        numeric_field_id = None
        group_field_id = None
        for field in fields:
            if field.get('dataType') in ['Integer', 'Float', 'Number']:
                numeric_field_id = field['@id']
            elif field.get('dataType') == 'Text' and not group_field_id:
                group_field_id = field['@id']
        if numeric_field_id in df.columns:
            plt.figure(figsize=(8,4))
            sns.histplot(df[numeric_field_id].dropna(), kde=True)
            plt.title(f"Distribution of {numeric_field_id}")
            plt.xlabel(numeric_field_id)
            plt.ylabel("Count")
            plt.show()
        if group_field_id and numeric_field_id and group_field_id in df.columns:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides detailed clinicopathological and molecular attributes for second primary colorectal cancer in survivors.
- Using `mlcroissant`, entities are easily referenced via their stable `@id`.
- Numeric fields (such as age or interval between diagnoses) can be filtered and normalized; categorical fields allow grouping and stratification analysis.
- Visualizations show distributions and relationships for further model building and research.

This notebook demonstrates dataset loading, structure exploration, basic wrangling, and plotting with reproducible IDs for fields and entities.